In [1]:
# USE THIS WHEN RUNNING ON COLAB

"""import kagglehub
import pandas as pd
import numpy as np

# Download latest version
path = kagglehub.dataset_download("s1nghayush/data-dump")

print("Path to dataset files:", path)"""

'import kagglehub\nimport pandas as pd\nimport numpy as np\n\n# Download latest version\npath = kagglehub.dataset_download("s1nghayush/data-dump")\n\nprint("Path to dataset files:", path)'

In [1]:
# USE THIS WHEN RUNNING LOCALLY
import pandas as pd
import numpy as np

df = pd.read_csv(r"D:\ED\FIFS_IPLungit\data\mbm\all_league_bat_final.csv")

In [2]:
df.head()

,match_id,date,inns,batter,batter_id,bat_team,bowl_team,runs,balls,fours,sixes,sr,isOut,wicketType,bowler,bowler_id,venue,league
0,211028,2005-06-13,1,ME Trescothick,21585,England,Australia,41,37,5,0,110.81,True,caught,A Symonds,7702.0,The Rose Bowl,T20IS
1,211028,2005-06-13,1,GO Jones,15501,England,Australia,19,14,4,0,135.71,True,caught,GD McGrath,6565.0,The Rose Bowl,T20IS
2,211028,2005-06-13,1,A Flintoff,12856,England,Australia,6,5,0,0,120.00,True,caught,MS Kasprowicz,6126.0,The Rose Bowl,T20IS
3,211028,2005-06-13,1,KP Pietersen,19296,England,Australia,34,18,3,1,188.89,True,caught,MJ Clarke,4578.0,The Rose Bowl,T20IS
4,211028,2005-06-13,1,MP Vaughan,22182,England,Australia,0,1,0,0,0.00,True,caught,A Symonds,7702.0,The Rose Bowl,T20IS


In [3]:
df['isOut'] = df['isOut'].replace('DNB', 'False')

In [4]:
df['isOut'] = df['isOut'].replace('True', True)
df['isOut'] = df['isOut'].replace('False', False)

In [5]:
# Define the conditions for each column
conditions = [
    (df['runs'] >= 25) & (df['runs'] < 50),
    (df['runs'] >= 50) & (df['runs'] < 75),
    (df['runs'] >= 75) & (df['runs'] < 100),
    (df['runs'] >= 100),
    (df['isOut'] & (df['runs'] == 0))  # 'isout' instead of 'isOut'
]

# Define the corresponding column names
column_names = ['25s', '50s', '75s', '100s', 'Duck']

# Initialize all new columns with 0
for col in column_names:
    df[col] = 0

# Assign 1 to the appropriate column based on conditions
for condition, col in zip(conditions, column_names):
    df.loc[condition, col] = 1


In [7]:
# Calculate batting fantasy points
df['batting_fp'] = (
    df['runs'] * 1 +  # Runs: +1 per run
    df['fours'] * 4 +  # Boundary Bonus: +4 per four
    df['sixes'] * 6 +  # Six Bonus: +6 per six
    df['25s'] * 4 +  # 25s: +4 pts
    df['50s'] * 8 +  # 50s: +8 pts
    df['75s'] * 12 +  # 75s: +12 pts
    df['100s'] * 16 +  # 100s: +16 pts
    df['Duck'] * -2  # Duck: -2 pts
)

# Add strike rate bonus/penalty
def strike_rate_points(row):
    if row['balls'] >= 10:  # Check if at least 10 balls were played
        if row['sr'] > 170:  # 'sr' instead of 'strikeRate'
            return 6
        elif 150.01 <= row['sr'] <= 170:
            return 4
        elif 130 <= row['sr'] <= 150:
            return 2
        elif 60 <= row['sr'] < 70:
            return -2
        elif 50 <= row['sr'] < 59.99:
            return -4
        elif row['sr'] < 50:
            return -6
    return 0  # Return 0 if fewer than 10 balls were played

# Apply strike rate points
df['strikeRate_points'] = df.apply(strike_rate_points, axis=1)
df['batting_fp'] += df['strikeRate_points']

# Drop the temporary strikeRate_points column if not needed
df.drop(columns=['strikeRate_points'], inplace=True)
df.head(10)


,match_id,date,inns,batter,batter_id,bat_team,bowl_team,runs,balls,fours,...,bowler,bowler_id,venue,league,25s,50s,75s,100s,Duck,batting_fp
0,211028,2005-06-13,1,ME Trescothick,21585,England,Australia,41,37,5,...,A Symonds,7702.0,The Rose Bowl,T20IS,1,0,0,0,0,65
1,211028,2005-06-13,1,GO Jones,15501,England,Australia,19,14,4,...,GD McGrath,6565.0,The Rose Bowl,T20IS,0,0,0,0,0,37
2,211028,2005-06-13,1,A Flintoff,12856,England,Australia,6,5,0,...,MS Kasprowicz,6126.0,The Rose Bowl,T20IS,0,0,0,0,0,6
3,211028,2005-06-13,1,KP Pietersen,19296,England,Australia,34,18,3,...,MJ Clarke,4578.0,The Rose Bowl,T20IS,1,0,0,0,0,62
4,211028,2005-06-13,1,MP Vaughan,22182,England,Australia,0,1,0,...,A Symonds,7702.0,The Rose Bowl,T20IS,0,0,0,0,1,-2
5,211028,2005-06-13,1,PD Collingwood,10772,England,Australia,46,26,5,...,GD McGrath,6565.0,The Rose Bowl,T20IS,1,0,0,0,0,88
6,211028,2005-06-13,1,AJ Strauss,20387,England,Australia,18,16,1,...,JN Gillespie,5392.0,The Rose Bowl,T20IS,0,0,0,0,0,22
7,211028,2005-06-13,1,VS Solanki,20286,England,Australia,9,5,1,...,GD McGrath,6565.0,The Rose Bowl,T20IS,0,0,0,0,0,13
8,211028,2005-06-13,1,D Gough,13411,England,Australia,0,0,0,...,NaN,NaN,The Rose Bowl,T20IS,0,0,0,0,0,0
9,211028,2005-06-13,1,J Lewis,16281,England,Australia,0,0,0,...,NaN,NaN,The Rose Bowl,T20IS,0,0,0,0,0,0


In [9]:
batting_fp = df.drop(columns=['inns', 'bat_team',  'isOut'])
batting_fp.head()

,match_id,date,batter,batter_id,bowl_team,runs,balls,fours,sixes,sr,...,bowler,bowler_id,venue,league,25s,50s,75s,100s,Duck,batting_fp
0,211028,2005-06-13,ME Trescothick,21585,Australia,41,37,5,0,110.81,...,A Symonds,7702.0,The Rose Bowl,T20IS,1,0,0,0,0,65
1,211028,2005-06-13,GO Jones,15501,Australia,19,14,4,0,135.71,...,GD McGrath,6565.0,The Rose Bowl,T20IS,0,0,0,0,0,37
2,211028,2005-06-13,A Flintoff,12856,Australia,6,5,0,0,120.00,...,MS Kasprowicz,6126.0,The Rose Bowl,T20IS,0,0,0,0,0,6
3,211028,2005-06-13,KP Pietersen,19296,Australia,34,18,3,1,188.89,...,MJ Clarke,4578.0,The Rose Bowl,T20IS,1,0,0,0,0,62
4,211028,2005-06-13,MP Vaughan,22182,Australia,0,1,0,0,0.00,...,A Symonds,7702.0,The Rose Bowl,T20IS,0,0,0,0,1,-2


In [10]:
batting_fp = df[['match_id', 'batter_id','batting_fp', 'date']].copy()
batting_fp

,match_id,batter_id,batting_fp,date
0,211028,21585,65,2005-06-13
1,211028,15501,37,2005-06-13
2,211028,12856,6,2005-06-13
3,211028,19296,62,2005-06-13
4,211028,22182,-2,2005-06-13
...,...,...,...,...
161021,1476798,1310415,0,2025-03-23
161022,1476798,574638,0,2025-03-23
161023,1476798,698317,0,2025-03-23
161024,1476798,524075,0,2025-03-23


In [11]:
min(batting_fp.date), max(batting_fp.date)

('2005-02-17', '2025-04-07')

In [12]:
batting_fp.to_csv(r"D:\ED\FIFS_IPLungit\data\processed_data\all_league_bat_fp.csv", index=False)